# Monte Carlo Simulation for European Put Option Pricing

This notebook implements Monte Carlo simulation for pricing European put options, compares with Black-Scholes theoretical values, and performs sensitivity and convergence analysis.

## Import Required Libraries

Import numpy, matplotlib, math, and scipy.stats for numerical computations, visualization, and statistical functions.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from math import log, sqrt, exp
from scipy.stats import norm

## Define Model Parameters

Set initial stock price $S_0$, strike price $K$, volatility $\sigma$, risk-free rate $r$, number of paths $M$, time to maturity $T$, and number of time steps $N$.

In [ ]:
np.random.seed(10000)  # for the sample to be reproducible

# Setting parameters
S0 = 100        # Initial stock price
K = 99          # Strike price
sigma = 0.2     # Volatility
r = 0.06        # Risk-free rate
M = 100000      # Number of simulation paths
T = 1           # Time to maturity (1 year)
N = 50          # Number of time steps
dt = T / N      # Time step size

## Monte Carlo Simulation Function

Implement the Monte Carlo simulation using geometric Brownian motion:
$$S_t = S_0 \exp\left((r - \frac{\sigma^2}{2})t + \sigma W_t\right)$$
to generate $M$ independent price paths with $N$ time steps.

In [ ]:
# Monte-carlo simulation with M independent paths and N time steps
def valueOption(S0, r, sigma, K, M, N, T): 
    S = S0 * np.exp(np.cumsum((r - 0.5 * sigma ** 2) * dt
    + sigma * np.sqrt(dt)
    * np.random.standard_normal((N + 1, M)), axis=0))
    S[0] = S0

    # Calculating the Monte Carlo estimator
    V0 = np.exp(-r * T) * np.sum(np.maximum(K-S[-1], 0)) / M
    return V0

## Calculate European Put Option Value

Compute the discounted expected payoff:
$$V_0 = e^{-rT} \mathbb{E}[\max(K - S_T, 0)]$$
using Monte Carlo estimation.

In [ ]:
put_option_value = valueOption(S0, r, sigma, K, M, N, T)

print('The European put option value is', put_option_value)

## Black-Scholes Theoretical Value

Implement the Black-Scholes formula for European put options using:
$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$$
$$d_2 = d_1 - \sigma\sqrt{T}$$
to compute the theoretical value for comparison.

In [ ]:
# Black-Scholes theoretical value
def d1(Stockprice, vol):
    return (log(Stockprice / K) + (r + vol ** 2 / 2) * T) / (vol * sqrt(T))


def d2(Stockprice, vol):
    return d1(Stockprice, vol) - vol * sqrt(T)


def bs_put(Stockprice, vol):
    BS_delta = norm.cdf(-d1(Stockprice, vol))
    Claim = K * exp(-r * T) * norm.cdf(-d2(Stockprice, vol)) - Stockprice * BS_delta 
    return BS_delta, Claim


BS_delta, Claim_BS = bs_put(S0, sigma)
print('The European put option value determined from Black-Scholes is:', Claim_BS)

## Convergence Analysis: Option Value vs Number of Paths

Plot the Monte Carlo estimator as a function of the number of simulation paths $M$ to visualize convergence to the true value.

In [ ]:
M_grid = [i for i in range(1, 300)]
C_grid = np.zeros(len(M_grid))
for i in range(len(M_grid)):
    C_grid[i] = valueOption(S0, r, sigma, K, M_grid[i], N, T)

plt.figure(figsize=(10, 6))
plt.plot(M_grid, C_grid, label='MC Estimator')
plt.axhline(y=Claim_BS, color='r', linestyle='--', label='Black-Scholes Value')
plt.xlabel('Number of Paths (M)')
plt.ylabel('European Put Option Value')
plt.title('Convergence of Monte Carlo Estimator')
plt.legend()
plt.show()

## Sensitivity Analysis: Option Value vs Volatility

Analyze how the European put option value changes with varying volatility $\sigma$ from 0 to 1.

In [ ]:
np.random.seed(1000)  # for reproducibility
M_sens = 200  # use fewer paths for sensitivity analysis

sigma_grid = np.linspace(0, 1, 100)
C_grid_sigma = np.zeros(len(sigma_grid))
for i in range(len(sigma_grid)):
    C_grid_sigma[i] = valueOption(S0, r, sigma_grid[i], K, M_sens, N, T)

plt.figure(figsize=(10, 6))
plt.plot(sigma_grid, C_grid_sigma)
plt.xlabel('Volatility ($\sigma$)')
plt.ylabel('European Put Option Value')
plt.title('Sensitivity to Volatility')
plt.show()

## Sensitivity Analysis: Option Value vs Strike Price

Examine the relationship between the option value and different strike prices $K$ ranging from 90 to 120.

In [ ]:
K_grid = [i for i in range(90, 120)]
C_grid_K = np.zeros(len(K_grid))
for i in range(len(K_grid)):
    C_grid_K[i] = valueOption(S0, r, sigma, K_grid[i], M_sens, N, T)

plt.figure(figsize=(10, 6))
plt.plot(K_grid, C_grid_K)
plt.xlabel('Strike Price (K)')
plt.ylabel('European Put Option Value')
plt.title('Sensitivity to Strike Price')
plt.show()

## Standard Error Estimation

Calculate the standard error of the Monte Carlo estimator using:
$$SE = \frac{\text{std}(\text{payoffs})}{\sqrt{M}}$$
to quantify estimation uncertainty.

In [ ]:
np.random.seed(10000)  # for reproducibility
M = 100000  # reset to large number of paths

# Monte-carlo simulation with M independent paths and N time steps (returning payoffs)
def valueOptionWithPayoff(S0, r, sigma, K, M, N, T): 
    S = S0 * np.exp(np.cumsum((r - 0.5 * sigma ** 2) * dt
    + sigma * np.sqrt(dt)
    * np.random.standard_normal((N + 1, M)), axis=0))
    S[0] = S0 
    payoff = np.maximum(K - S[-1], 0)
    # Calculating the Monte Carlo estimator
    V0 = np.exp(-r * T) * np.sum(payoff) / M
    return V0, payoff

V0, payoff = valueOptionWithPayoff(S0, r, sigma, K, M, N, T)
SE = np.std(payoff) / np.sqrt(M)

print("European Put option value is {0:.4f} with standard error {1:.4f}".format(V0, SE))
print("95% Confidence Interval: [{0:.4f}, {1:.4f}]".format(V0 - 1.96*SE, V0 + 1.96*SE))